# **WEEK 6 — State Space Models and the Kalman Filter**

>Jerald B. Bongalos | PhD in Data Science | Asian Institute of Management

---

*Reference:*
>Shumway, R. H., & Stoffer, D. S. *Time Series Analysis and Its Applications*
>(Ch 6.1–6.5: State Space Models)

---

**Purpose of this Notebook**

This notebook introduces the **state space framework** — a unifying representation
that encompasses ARIMA, structural decomposition, and dynamic linear models
as special cases. The **Kalman filter** provides optimal recursive estimation
within this framework.

**The goal is to understand:**

- how state space models separate observation from hidden dynamics,
- why the Kalman filter is optimal (minimum mean square error),
- how prediction, filtering, and smoothing differ,
- how ARIMA models map into state space form,
- how structural time series models decompose trend, seasonality, and noise,
- and how the Kalman filter handles missing data and computes exact likelihoods.

**Learning Outcomes**

By the end of this notebook, you should be able to:

1. Write the general linear state space model (state + observation equations)
2. Derive and implement the Kalman filter prediction and update steps
3. Explain the difference between filtering, prediction, and smoothing
4. Convert AR(1) and ARIMA models to state space form
5. Formulate structural models (local level, local linear trend)
6. Use the Kalman filter to handle missing observations
7. Explain how exact likelihood is computed via the innovation form

---

**Notebook Structure**

| Part | Topic | Type | Priority |
|------|-------|------|----------|
| 1 | State Space Representation | Theory + Simulation | CRITICAL |
| 2 | The Kalman Filter: Predict and Update | Theory + Simulation | CRITICAL |
| 3 | Filtering, Prediction, and Smoothing | Theory + Simulation | CRITICAL |
| 4 | ARIMA in State Space Form | Theory + Simulation | CRITICAL |
| 5 | Structural Time Series Models | Theory + Simulation | MEDIUM |
| 6 | Missing Data and Exact Likelihood | Theory + Simulation | CRITICAL |
| 7 | Synthesis (Exam-Ready) | Summary | — |
| 8 | Self-Test Questions | Exam Preparation | — |


In [ ]:
# ============================================================
# SETUP
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve
from statsmodels.tsa.statespace.structural import UnobservedComponents
from statsmodels.tsa.stattools import acf

np.random.seed(123)

print("Setup complete.")


## **PART 1 — State Space Representation**

> **Why this part matters**
>
> Every model we have studied — AR, MA, ARMA, ARIMA, structural decomposition —
> can be written as a **state space model**. The state space framework separates
> *what we observe* from *what drives the system*, enabling a unified treatment
> of estimation, forecasting, and missing data.

---

### **1.1 The General Linear State Space Model**

**State equation** (hidden dynamics):

$$
\mathbf{x}_{t} = \boldsymbol{\Phi}\,\mathbf{x}_{t-1} + \mathbf{w}_{t}, \qquad \mathbf{w}_{t} \sim N(\mathbf{0}, \mathbf{Q})
$$

**Observation equation** (what we measure):

$$
y_{t} = \mathbf{A}'\,\mathbf{x}_{t} + v_{t}, \qquad v_{t} \sim N(0, R)
$$

where:
- $\mathbf{x}_t$ is the $p \times 1$ **state vector** (unobserved)
- $y_t$ is the scalar (or vector) **observation**
- $\boldsymbol{\Phi}$ is the $p \times p$ **state transition matrix**
- $\mathbf{A}$ is the $p \times 1$ **observation matrix**
- $\mathbf{Q}$ is the $p \times p$ state noise covariance
- $R$ is the observation noise variance
- $\mathbf{w}_t$ and $v_t$ are mutually independent

---

### **1.2 Why State Space?**

The state space framework provides:

- **Unified representation** of ARMA, structural, and regression models
- **Recursive estimation** via the Kalman filter (no matrix inversion of growing dimension)
- **Missing data handling** — simply skip the update step when $y_t$ is missing
- **Exact likelihood** computation via the innovation form
- **Time-varying parameters** — $\boldsymbol{\Phi}$, $\mathbf{A}$, $\mathbf{Q}$, $R$ can change over time

---

### **1.3 Simplest Example: Local Level Model**

$$
x_t = x_{t-1} + w_t, \qquad w_t \sim N(0, \sigma_w^2)
$$
$$
y_t = x_t + v_t, \qquad v_t \sim N(0, \sigma_v^2)
$$

The **state** $x_t$ is a random walk (the "true level"), and $y_t$ is a noisy observation.
The Kalman filter estimates $x_t$ given the observations $y_1, \ldots, y_t$.

> **Exam-safe statement:** "The state space model separates the observation process from
> the latent dynamics. The Kalman filter provides the optimal (MMSE) recursive estimator
> for the hidden state."


In [ ]:
# ============================================================
# PART 1: Local Level Model — Simulation
# ============================================================

np.random.seed(123)
n = 200

sigma_w = 0.5   # state noise (random walk step size)
sigma_v = 1.5   # observation noise

# --- Simulate local level model ---
x_true = np.zeros(n)
y_obs = np.zeros(n)

for t in range(1, n):
    x_true[t] = x_true[t-1] + np.random.normal(0, sigma_w)

y_obs = x_true + np.random.normal(0, sigma_v, n)

# --- Plot ---
fig, ax = plt.subplots(1, 1, figsize=(14, 4))
ax.plot(y_obs, linewidth=0.6, alpha=0.7, color="#95a5a6", label="Observations $y_t$")
ax.plot(x_true, linewidth=2, color="#e74c3c", label="True state $x_t$ (unobserved)")
ax.set_title("Local Level Model: Noisy Observations of a Random Walk", fontsize=13)
ax.set_xlabel("$t$")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Signal-to-noise ratio
q = sigma_w**2 / sigma_v**2
print(f"State noise variance (Q):       sigma_w^2 = {sigma_w**2}")
print(f"Observation noise variance (R):  sigma_v^2 = {sigma_v**2}")
print(f"Signal-to-noise ratio Q/R:       {q:.4f}")
print(f"\nSmall Q/R -> observations are noisy, heavy smoothing needed")
print(f"Large Q/R -> state changes fast, filter tracks observations closely")


## **PART 2 — The Kalman Filter: Predict and Update**

> **Why this part matters**
>
> The Kalman filter is the **optimal recursive estimator** for the state vector
> in a linear Gaussian state space model. It alternates between two steps:
> **prediction** (project forward) and **update** (incorporate new observation).

---

### **2.1 Notation**

- $\mathbf{x}_{t|s}^{} = \mathbb{E}[\mathbf{x}_t \mid y_1, \ldots, y_s]$: state estimate given data up to time $s$
- $\mathbf{P}_{t|s}^{} = \mathrm{Cov}(\mathbf{x}_t - \mathbf{x}_{t|s}^{})$: estimation error covariance

Key cases:
- $s = t-1$: **prediction** (one-step-ahead)
- $s = t$: **filtering** (current estimate)
- $s = n > t$: **smoothing** (retrospective, uses all data)

---

### **2.2 Prediction Step**

Given the filtered estimate at $t-1$:

$$
\mathbf{x}_{t|t-1}^{} = \boldsymbol{\Phi}\,\mathbf{x}_{t-1|t-1}^{}
$$

$$
\mathbf{P}_{t|t-1}^{} = \boldsymbol{\Phi}\,\mathbf{P}_{t-1|t-1}^{}\,\boldsymbol{\Phi}' + \mathbf{Q}
$$

---

### **2.3 Innovation**

The **innovation** (prediction error) is:

$$
\nu_t = y_t - \mathbf{A}'\,\mathbf{x}_{t|t-1}^{}
$$

with variance:

$$
S_t = \mathbf{A}'\,\mathbf{P}_{t|t-1}^{}\,\mathbf{A} + R
$$

---

### **2.4 Update Step (Filtering)**

The **Kalman gain**:

$$
\mathbf{K}_t = \mathbf{P}_{t|t-1}^{}\,\mathbf{A}\,S_t^{-1}
$$

Updated state estimate:

$$
\mathbf{x}_{t|t}^{} = \mathbf{x}_{t|t-1}^{} + \mathbf{K}_t\,\nu_t
$$

Updated error covariance:

$$
\mathbf{P}_{t|t}^{} = (\mathbf{I} - \mathbf{K}_t\,\mathbf{A}')\,\mathbf{P}_{t|t-1}^{}
$$

---

### **2.5 Intuition**

The Kalman gain $\mathbf{K}_t$ balances two sources of information:

- **Prior** (prediction from state equation): weight $\propto 1/\mathbf{P}_{t|t-1}$
- **Observation** (new data $y_t$): weight $\propto 1/R$

If $R$ is large (noisy observations): $\mathbf{K}_t$ is small → filter trusts the prediction.
If $\mathbf{Q}$ is large (volatile state): $\mathbf{P}_{t|t-1}$ is large → $\mathbf{K}_t$ is large → filter trusts the observation.

> **Exam-critical:** "The Kalman filter is the MMSE estimator for linear Gaussian state space models.
> The Kalman gain optimally weights prior prediction against new observation."


In [ ]:
# ============================================================
# PART 2: Kalman Filter — From-Scratch Implementation
# ============================================================

def kalman_filter_local_level(y, sigma_w, sigma_v, x0=0, P0=1.0):
    """
    Kalman filter for the local level model:
        x_t = x_{t-1} + w_t,  w_t ~ N(0, sigma_w^2)
        y_t = x_t + v_t,      v_t ~ N(0, sigma_v^2)

    Returns filtered states, predicted states, Kalman gains, innovations.
    """
    n = len(y)
    Q = sigma_w**2
    R = sigma_v**2

    # Storage
    x_pred = np.zeros(n)     # x_{t|t-1}
    P_pred = np.zeros(n)     # P_{t|t-1}
    x_filt = np.zeros(n)     # x_{t|t}
    P_filt = np.zeros(n)     # P_{t|t}
    K = np.zeros(n)          # Kalman gain
    nu = np.zeros(n)         # Innovation
    S = np.zeros(n)          # Innovation variance
    log_lik = 0.0

    # Initialize
    x_filt_prev = x0
    P_filt_prev = P0

    for t in range(n):
        # --- Prediction ---
        x_pred[t] = x_filt_prev           # Phi = 1 for local level
        P_pred[t] = P_filt_prev + Q

        # --- Innovation ---
        if np.isnan(y[t]):
            # Missing observation: skip update
            x_filt[t] = x_pred[t]
            P_filt[t] = P_pred[t]
            K[t] = 0
            nu[t] = np.nan
            S[t] = np.nan
        else:
            S[t] = P_pred[t] + R           # A = 1 for local level
            nu[t] = y[t] - x_pred[t]

            # --- Kalman gain ---
            K[t] = P_pred[t] / S[t]

            # --- Update ---
            x_filt[t] = x_pred[t] + K[t] * nu[t]
            P_filt[t] = (1 - K[t]) * P_pred[t]

            # --- Log-likelihood contribution ---
            log_lik += -0.5 * (np.log(2*np.pi) + np.log(S[t]) + nu[t]**2 / S[t])

        x_filt_prev = x_filt[t]
        P_filt_prev = P_filt[t]

    return {
        "x_pred": x_pred, "P_pred": P_pred,
        "x_filt": x_filt, "P_filt": P_filt,
        "K": K, "nu": nu, "S": S, "log_lik": log_lik
    }

# --- Run Kalman filter on simulated data ---
kf = kalman_filter_local_level(y_obs, sigma_w, sigma_v, x0=0, P0=10.0)

# --- Plot: True state, observations, filtered estimate ---
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Panel A: Filtering
ax = axes[0]
ax.plot(y_obs, linewidth=0.5, alpha=0.5, color="#95a5a6", label="Observations $y_t$")
ax.plot(x_true, linewidth=2, color="#e74c3c", label="True state $x_t$")
ax.plot(kf["x_filt"], linewidth=2, color="#2980b9", label="Filtered $\\hat{x}_{t|t}$")
# 95% CI
ci = 1.96 * np.sqrt(kf["P_filt"])
ax.fill_between(range(n), kf["x_filt"] - ci, kf["x_filt"] + ci,
                alpha=0.2, color="#2980b9", label="95% CI")
ax.set_title("Kalman Filter: Filtered State Estimate", fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel B: Kalman gain convergence
ax = axes[1]
ax.plot(kf["K"], linewidth=1.5, color="#27ae60")
ax.axhline(kf["K"][-1], color="red", linestyle="--", alpha=0.7,
           label=f"Steady-state $K = {kf['K'][-1]:.4f}$")
ax.set_title("Kalman Gain Convergence", fontsize=13)
ax.set_xlabel("$t$")
ax.set_ylabel("$K_t$")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Steady-state Kalman gain (analytical) ---
# For local level: K_ss = (sqrt(Q^2 + 4QR) - Q) / (2R)  ... from algebraic Riccati
K_ss_analytical = (np.sqrt(sigma_w**4 + 4*sigma_w**2*sigma_v**2) - sigma_w**2) / (2*sigma_v**2)
print(f"Steady-state Kalman gain:")
print(f"  Numerical (last K_t):  {kf['K'][-1]:.6f}")
print(f"  Analytical:            {K_ss_analytical:.6f}")
print(f"\nLog-likelihood: {kf['log_lik']:.4f}")


## **PART 3 — Filtering, Prediction, and Smoothing**

> **Why this part matters**
>
> The Kalman filter produces the **filtered** estimate $\mathbf{x}_{t|t}$ (using data up to $t$).
> But we often want other quantities:
>
> - **Prediction** $\mathbf{x}_{t+h|t}$: forecast future states
> - **Smoothing** $\mathbf{x}_{t|n}$: re-estimate past states using ALL data (retrospective)

---

### **3.1 Summary**

| Estimator | Notation | Data used | Use case |
|---|---|---|---|
| Prediction | $\mathbf{x}_{t+h\|t}$ | $y_1, \ldots, y_t$ | Forecasting |
| Filtering | $\mathbf{x}_{t\|t}$ | $y_1, \ldots, y_t$ | Real-time tracking |
| Smoothing | $\mathbf{x}_{t\|n}$ | $y_1, \ldots, y_n$ | Retrospective analysis |

---

### **3.2 Smoothing**

The **Rauch–Tung–Striebel (RTS) smoother** runs backward after the forward Kalman filter:

$$
\mathbf{J}_t = \mathbf{P}_{t|t}\,\boldsymbol{\Phi}'\,\mathbf{P}_{t+1|t}^{-1}
$$

$$
\mathbf{x}_{t|n} = \mathbf{x}_{t|t} + \mathbf{J}_t(\mathbf{x}_{t+1|n} - \mathbf{x}_{t+1|t})
$$

$$
\mathbf{P}_{t|n} = \mathbf{P}_{t|t} + \mathbf{J}_t(\mathbf{P}_{t+1|n} - \mathbf{P}_{t+1|t})\mathbf{J}_t'
$$

**Key property:** $\mathbf{P}_{t|n} \leq \mathbf{P}_{t|t} \leq \mathbf{P}_{t|t-1}$

Smoothed estimates are always at least as precise as filtered estimates.

> **Exam-safe statement:** "Filtering uses data up to time $t$; smoothing uses all $n$ observations.
> Smoothing always has smaller error covariance than filtering."


In [ ]:
# ============================================================
# PART 3: RTS Smoother + Comparison
# ============================================================

def rts_smoother_local_level(kf_result):
    """RTS smoother for local level model (Phi=1, A=1)."""
    x_filt = kf_result["x_filt"]
    P_filt = kf_result["P_filt"]
    x_pred = kf_result["x_pred"]
    P_pred = kf_result["P_pred"]
    n = len(x_filt)

    x_smooth = np.zeros(n)
    P_smooth = np.zeros(n)

    # Initialize at t=n-1
    x_smooth[-1] = x_filt[-1]
    P_smooth[-1] = P_filt[-1]

    # Backward pass
    for t in range(n-2, -1, -1):
        J_t = P_filt[t] / P_pred[t+1]  # Phi=1 for local level
        x_smooth[t] = x_filt[t] + J_t * (x_smooth[t+1] - x_pred[t+1])
        P_smooth[t] = P_filt[t] + J_t**2 * (P_smooth[t+1] - P_pred[t+1])

    return x_smooth, P_smooth

x_smooth, P_smooth = rts_smoother_local_level(kf)

# --- Compare filtered vs smoothed ---
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Panel A: State estimates
ax = axes[0]
ax.plot(y_obs, linewidth=0.4, alpha=0.4, color="#95a5a6", label="Observations")
ax.plot(x_true, linewidth=2, color="#e74c3c", label="True state")
ax.plot(kf["x_filt"], linewidth=1.5, color="#2980b9", linestyle="--", label="Filtered $\\hat{x}_{t|t}$")
ax.plot(x_smooth, linewidth=1.5, color="#27ae60", label="Smoothed $\\hat{x}_{t|n}$")
ax.set_title("Filtered vs Smoothed State Estimates", fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel B: Error covariance comparison
ax = axes[1]
ax.plot(kf["P_pred"], linewidth=1.5, color="#e67e22", label="Prediction $P_{t|t-1}$")
ax.plot(kf["P_filt"], linewidth=1.5, color="#2980b9", label="Filtering $P_{t|t}$")
ax.plot(P_smooth, linewidth=1.5, color="#27ae60", label="Smoothing $P_{t|n}$")
ax.set_title("Error Covariance: Prediction $\\geq$ Filtering $\\geq$ Smoothing", fontsize=13)
ax.set_xlabel("$t$")
ax.set_ylabel("$P$")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- MSE comparison ---
mse_filt = np.mean((kf["x_filt"] - x_true)**2)
mse_smooth = np.mean((x_smooth - x_true)**2)
mse_obs = np.mean((y_obs - x_true)**2)
print(f"MSE comparison:")
print(f"  Raw observations:     {mse_obs:.4f}")
print(f"  Kalman filtered:      {mse_filt:.4f}")
print(f"  RTS smoothed:         {mse_smooth:.4f}")
print(f"\n  Smoothing < Filtering < Raw (as theory predicts)")


## **PART 4 — ARIMA in State Space Form**

> **Why this part matters**
>
> ARIMA models can be written in state space form. This is not merely academic —
> it enables exact likelihood computation, missing data handling, and provides
> the foundation for how `statsmodels` and `R`'s `forecast` package fit ARIMA internally.

---

### **4.1 AR(1) in State Space**

The AR(1) model $X_t = \phi X_{t-1} + \varepsilon_t$ has trivial state space form:

$$
x_t = \phi\, x_{t-1} + w_t, \qquad y_t = x_t
$$

Here $\boldsymbol{\Phi} = \phi$, $\mathbf{A} = 1$, $Q = \sigma^2$, $R = 0$ (no observation noise).

With observation noise ($R > 0$), this becomes the AR(1) plus noise model —
the state is the "true signal" observed with measurement error.

---

### **4.2 ARMA(1,1) in State Space**

$X_t = \phi X_{t-1} + \varepsilon_t + \theta\varepsilon_{t-1}$ requires a 2-dimensional state:

$$
\mathbf{x}_t = \begin{pmatrix} X_t \\ \theta\varepsilon_t \end{pmatrix}, \quad
\boldsymbol{\Phi} = \begin{pmatrix} \phi & 1 \\ 0 & 0 \end{pmatrix}, \quad
\mathbf{A} = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \quad
\mathbf{Q} = \begin{pmatrix} \sigma^2 & \theta\sigma^2 \\ \theta\sigma^2 & \theta^2\sigma^2 \end{pmatrix}
$$

---

### **4.3 General ARIMA($p,d,q$)**

Any ARIMA($p,d,q$) can be put in state space form with state dimension $r = \max(p, q+1)$.
The details are model-specific, but the key insight is:

- **State dimension** depends on the ARIMA orders
- **Transition matrix** $\boldsymbol{\Phi}$ encodes the AR dynamics
- **Observation matrix** $\mathbf{A}$ selects the observable component
- **State noise** $\mathbf{Q}$ encodes the MA structure

> **Exam-safe statement:** "Any ARIMA model has a state space representation.
> Fitting ARIMA via state space and Kalman filter gives exact (not conditional) MLE."


In [ ]:
# ============================================================
# PART 4: AR(1) in State Space — Kalman Filter vs Direct
# ============================================================

np.random.seed(42)

# --- Simulate AR(1) with observation noise ---
phi = 0.8
sigma_w = 1.0    # state noise
sigma_v = 0.5    # observation noise
n = 300

x_ar = np.zeros(n)
for t in range(1, n):
    x_ar[t] = phi * x_ar[t-1] + np.random.normal(0, sigma_w)

y_ar = x_ar + np.random.normal(0, sigma_v, n)

# --- Kalman filter for AR(1) + noise ---
def kalman_ar1(y, phi, sigma_w, sigma_v, x0=0, P0=10.0):
    """Kalman filter for AR(1) + observation noise."""
    n = len(y)
    Q = sigma_w**2
    R = sigma_v**2
    x_filt = np.zeros(n)
    P_filt = np.zeros(n)
    x_pred = np.zeros(n)

    xf, Pf = x0, P0
    for t in range(n):
        # Predict
        xp = phi * xf
        Pp = phi**2 * Pf + Q
        x_pred[t] = xp
        # Update
        if not np.isnan(y[t]):
            S = Pp + R
            K = Pp / S
            x_filt[t] = xp + K * (y[t] - xp)
            P_filt[t] = (1 - K) * Pp
        else:
            x_filt[t] = xp
            P_filt[t] = Pp
        xf, Pf = x_filt[t], P_filt[t]
    return x_filt, P_filt, x_pred

x_filt_ar, P_filt_ar, x_pred_ar = kalman_ar1(y_ar, phi, sigma_w, sigma_v)

fig, ax = plt.subplots(1, 1, figsize=(14, 4))
ax.plot(y_ar, linewidth=0.4, alpha=0.5, color="#95a5a6", label="Observations $y_t$")
ax.plot(x_ar, linewidth=2, color="#e74c3c", label="True AR(1) state")
ax.plot(x_filt_ar, linewidth=1.5, color="#2980b9", label="Kalman filtered")
ci = 1.96 * np.sqrt(P_filt_ar)
ax.fill_between(range(n), x_filt_ar - ci, x_filt_ar + ci, alpha=0.15, color="#2980b9")
ax.set_title(f"AR(1) + Noise: Kalman Filter ($\\phi={phi}$, $\\sigma_w={sigma_w}$, $\\sigma_v={sigma_v}$)", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

mse_raw = np.mean((y_ar - x_ar)**2)
mse_kf = np.mean((x_filt_ar - x_ar)**2)
print(f"MSE (raw observations):   {mse_raw:.4f}")
print(f"MSE (Kalman filtered):    {mse_kf:.4f}")
print(f"Improvement:              {(1 - mse_kf/mse_raw)*100:.1f}%")


## **PART 5 — Structural Time Series Models**

> **Why this part matters**
>
> Structural models decompose a time series into interpretable components —
> **trend**, **seasonality**, **cycle**, and **irregular** — each modeled as
> a latent state. The Kalman filter estimates all components simultaneously.

---

### **5.1 Local Level Model** (Review)

$$
\mu_t = \mu_{t-1} + \eta_t, \qquad y_t = \mu_t + \varepsilon_t
$$

Level is a random walk; observation is noisy.

---

### **5.2 Local Linear Trend Model**

$$
\mu_t = \mu_{t-1} + \nu_{t-1} + \eta_t \qquad \text{(level)}
$$
$$
\nu_t = \nu_{t-1} + \zeta_t \qquad \text{(slope/trend)}
$$
$$
y_t = \mu_t + \varepsilon_t \qquad \text{(observation)}
$$

The state vector is $\mathbf{x}_t = (\mu_t, \nu_t)'$ — both level and slope are stochastic.

---

### **5.3 Adding Seasonality**

A stochastic seasonal component $\gamma_t$ (period $s$) satisfies:

$$
\gamma_t = -\sum_{j=1}^{s-1} \gamma_{t-j} + \omega_t
$$

This constrains the seasonal effects to sum to approximately zero over a full cycle.
The state vector now includes $s-1$ seasonal states.

---

### **5.4 Full Structural Model**

$$
y_t = \mu_t + \gamma_t + \varepsilon_t
$$

with separate state equations for level, slope, and seasonality.
The Kalman filter estimates all components jointly.

> **Exam-safe statement:** "Structural models provide interpretable decomposition.
> The Kalman filter estimates all unobserved components simultaneously via
> optimal recursive filtering."


In [ ]:
# ============================================================
# PART 5: Structural Models via statsmodels UnobservedComponents
# ============================================================

np.random.seed(123)
n = 240
t = np.arange(n)
idx = pd.date_range("2005-01-01", periods=n, freq="MS")

# --- Simulate: trend + seasonality + noise ---
trend = 50 + 0.1 * t + np.cumsum(np.random.normal(0, 0.3, n))
season = 5 * np.sin(2*np.pi*t/12) + 2 * np.cos(2*np.pi*t/6)
noise = np.random.normal(0, 2.0, n)
y_struct = trend + season + noise

y_series = pd.Series(y_struct, index=idx, name="y")

# --- Fit structural model: local linear trend + stochastic seasonal ---
mod = UnobservedComponents(y_series, level="local linear trend",
                            seasonal=12, stochastic_seasonal=True)
res = mod.fit(disp=False)
print("=== STRUCTURAL MODEL FIT ===")
print(res.summary().tables[1])

# --- Extract components ---
level_est = res.level["smoothed"]
trend_est = res.trend["smoothed"]
season_est = res.seasonal["smoothed"]
resid_est = res.resid

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(y_series, linewidth=0.8, color="#95a5a6", label="Observed")
axes[0].plot(level_est, linewidth=2, color="#2980b9", label="Estimated level")
axes[0].set_title("Observed Series + Estimated Level", fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(trend_est, linewidth=2, color="#e74c3c")
axes[1].set_title("Estimated Slope (Trend Growth Rate)", fontsize=12)
axes[1].grid(True, alpha=0.3)

axes[2].plot(season_est, linewidth=1.5, color="#8e44ad")
axes[2].set_title("Estimated Seasonal Component", fontsize=12)
axes[2].grid(True, alpha=0.3)

axes[3].plot(resid_est, linewidth=0.8, color="#27ae60")
axes[3].axhline(0, color="gray", linestyle="--", alpha=0.5)
axes[3].set_title("Residuals (Irregular Component)", fontsize=12)
axes[3].set_xlabel("Date")
axes[3].grid(True, alpha=0.3)

plt.suptitle("Structural Decomposition via Kalman Filter", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# --- Residual check ---
a_resid = acf(resid_est.dropna(), nlags=24, fft=True)
ci = 1.96 / np.sqrt(len(resid_est.dropna()))
plt.figure(figsize=(10, 3))
plt.stem(range(len(a_resid)), a_resid, basefmt=" ")
plt.axhline(ci, color="red", linestyle="--", alpha=0.5)
plt.axhline(-ci, color="red", linestyle="--", alpha=0.5)
plt.title("Residual ACF — Structural Model", fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Level: captures the slowly evolving mean")
print("  Slope: captures acceleration/deceleration in trend")
print("  Seasonal: recovers the annual + semi-annual pattern")
print("  Residuals: should be approximately white noise")


## **PART 6 — Missing Data and Exact Likelihood**

> **Why this part matters**
>
> Two of the Kalman filter's most powerful practical advantages:
>
> 1. **Missing data**: simply skip the update step when $y_t$ is unobserved
> 2. **Exact likelihood**: the innovation form gives an exact log-likelihood
>    without approximations or conditional tricks

---

### **6.1 Missing Data Handling**

When $y_t$ is missing:
- **Prediction step** proceeds normally: $\mathbf{x}_{t|t-1}$, $\mathbf{P}_{t|t-1}$
- **Update step is skipped**: set $\mathbf{x}_{t|t} = \mathbf{x}_{t|t-1}$, $\mathbf{P}_{t|t} = \mathbf{P}_{t|t-1}$

The filter carries forward with increased uncertainty. When the next observation
arrives, the Kalman gain automatically gives it more weight.

> No imputation is needed. Missing values are handled **within** the filter.

---

### **6.2 Innovation Form of the Log-Likelihood**

The exact log-likelihood is computed from the **innovations** $\nu_t = y_t - \mathbf{A}'\mathbf{x}_{t|t-1}$:

$$
\ln L = -\frac{1}{2}\sum_{t=1}^{n} \left[\ln(2\pi) + \ln(S_t) + \frac{\nu_t^2}{S_t}\right]
$$

where $S_t = \mathbf{A}'\mathbf{P}_{t|t-1}\mathbf{A} + R$ is the innovation variance.

This is **exact** (not conditional on initial values) when the initial state distribution
is properly specified, and it only requires quantities already computed by the Kalman filter.

---

### **6.3 Why This Matters for ARIMA**

- Classical ARIMA fitting uses **conditional sum of squares** (approximate MLE)
- State space ARIMA uses **exact MLE** via the Kalman filter
- `statsmodels.tsa.statespace` and R's `stats::arima()` use this internally
- Exact MLE is more efficient, especially for short series

> **Exam-safe statement:** "The Kalman filter computes the exact log-likelihood via the innovation form.
> Missing data require no imputation — the update step is simply skipped."


In [ ]:
# ============================================================
# PART 6: Missing Data — Kalman Filter Handles Gaps
# ============================================================

np.random.seed(123)

# --- Create data with missing values ---
y_missing = y_obs.copy()
missing_idx = np.random.choice(np.arange(20, n-20), size=40, replace=False)
missing_idx.sort()
y_missing[missing_idx] = np.nan

print(f"Total observations: {n}")
print(f"Missing values:     {np.isnan(y_missing).sum()} ({np.isnan(y_missing).mean()*100:.0f}%)")

# --- Run Kalman filter on complete and incomplete data ---
kf_complete = kalman_filter_local_level(y_obs, sigma_w, sigma_v, x0=0, P0=10.0)
kf_missing = kalman_filter_local_level(y_missing, sigma_w, sigma_v, x0=0, P0=10.0)

# --- Plot comparison ---
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Panel A: Complete data
ax = axes[0]
ax.plot(y_obs, linewidth=0.4, alpha=0.4, color="#95a5a6", label="Observations")
ax.plot(x_true, linewidth=2, color="#e74c3c", label="True state")
ax.plot(kf_complete["x_filt"], linewidth=1.5, color="#2980b9", label="Filtered (complete)")
ax.set_title("Kalman Filter — Complete Data", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel B: Missing data
ax = axes[1]
observed_mask = ~np.isnan(y_missing)
ax.scatter(np.where(observed_mask)[0], y_missing[observed_mask],
           s=3, alpha=0.4, color="#95a5a6", label="Observed points")
ax.scatter(missing_idx, x_true[missing_idx], s=15, color="red",
           marker="x", zorder=5, label="Missing locations")
ax.plot(x_true, linewidth=2, color="#e74c3c", label="True state")
ax.plot(kf_missing["x_filt"], linewidth=1.5, color="#2980b9", label="Filtered (with gaps)")
ci = 1.96 * np.sqrt(kf_missing["P_filt"])
ax.fill_between(range(n), kf_missing["x_filt"] - ci, kf_missing["x_filt"] + ci,
                alpha=0.15, color="#2980b9")
ax.set_title("Kalman Filter — 20% Missing Data (no imputation needed)", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Uncertainty grows during gaps ---
fig, ax = plt.subplots(1, 1, figsize=(14, 3))
ax.plot(kf_complete["P_filt"], linewidth=1.5, color="#2980b9", label="$P_{t|t}$ (complete)")
ax.plot(kf_missing["P_filt"], linewidth=1.5, color="#e74c3c", label="$P_{t|t}$ (with gaps)")
for idx_m in missing_idx:
    ax.axvline(idx_m, color="gray", alpha=0.1, linewidth=0.5)
ax.set_title("Filtering Covariance: Uncertainty Grows During Missing Periods", fontsize=12)
ax.set_xlabel("$t$")
ax.set_ylabel("$P_{t|t}$")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- MSE comparison ---
mse_complete = np.mean((kf_complete["x_filt"] - x_true)**2)
mse_missing = np.mean((kf_missing["x_filt"] - x_true)**2)
print(f"\nMSE (complete data):  {mse_complete:.4f}")
print(f"MSE (20% missing):    {mse_missing:.4f}")
print(f"Degradation:          {(mse_missing/mse_complete - 1)*100:.1f}%")
print(f"\nLog-likelihood (complete): {kf_complete['log_lik']:.2f}")
print(f"Log-likelihood (missing):  {kf_missing['log_lik']:.2f}")
print(f"\nKey: the filter gracefully degrades — uncertainty increases in gaps")
print(f"but recovers as new observations arrive.")


## **PART 7 — Synthesis (Exam-Ready)**

> **Core points you must be able to state without hesitation:**

---

### **State Space Representation**

- The state space model has two equations: **state** (hidden dynamics) and **observation** (measurement)
- It unifies ARIMA, structural decomposition, and dynamic linear models into one framework
- The state vector captures **all relevant information** from the past (Markov property)
- The model can be time-varying — parameters $\boldsymbol{\Phi}_t$, $\mathbf{A}_t$, $\mathbf{Q}_t$, $R_t$ can change

### **Kalman Filter**

- The Kalman filter alternates between **prediction** and **update** steps
- It is the **MMSE** (minimum mean square error) linear estimator under Gaussian assumptions
- The **Kalman gain** $K_t$ optimally balances prior prediction vs new observation
- Large observation noise $R$ → small $K_t$ → filter trusts the model
- Large state noise $Q$ → large $K_t$ → filter trusts the data
- The Kalman gain **converges** to a steady state (from the algebraic Riccati equation)

### **Filtering vs Prediction vs Smoothing**

- **Filtering** $\mathbf{x}_{t|t}$: uses $y_1, \ldots, y_t$ (real-time)
- **Prediction** $\mathbf{x}_{t+h|t}$: forecasting future states
- **Smoothing** $\mathbf{x}_{t|n}$: uses all data $y_1, \ldots, y_n$ (retrospective)
- Always: $P_{\text{smooth}} \leq P_{\text{filter}} \leq P_{\text{predict}}$

### **ARIMA in State Space**

- Any ARIMA model has a state space representation
- State space fitting gives **exact MLE** (not conditional)
- This is how modern software (statsmodels, R) fits ARIMA internally

### **Structural Models**

- Decompose series into trend, seasonality, cycle, irregular
- Each component is a latent state estimated by the Kalman filter
- Provides **interpretable decomposition** with uncertainty quantification

### **Missing Data and Likelihood**

- Missing values: skip the update step — no imputation needed
- Uncertainty grows during gaps and shrinks when observations resume
- Exact log-likelihood computed from innovations: $\ln L = -\frac{1}{2}\sum[\ln S_t + \nu_t^2/S_t] + \text{const}$


## **PART 8 — Self-Test: Exam Preparation Questions**

> Work through these without looking at notes first.

---

### **Conceptual Questions (Oral Exam Style)**

**Q1.** Write the general linear state space model. What are the state equation and observation equation? What do $\boldsymbol{\Phi}$, $\mathbf{A}$, $\mathbf{Q}$, $R$ represent?

**Q2.** Derive the Kalman filter prediction and update steps for the local level model. What is the Kalman gain and what does it optimize?

**Q3.** Explain the difference between filtering, prediction, and smoothing. Which gives the smallest estimation error? Why?

**Q4.** Your colleague says "the Kalman filter requires Gaussian assumptions." Is this strictly true? What optimality property does the filter have without Gaussianity?

**Q5.** Write the AR(1) model $X_t = \phi X_{t-1} + \varepsilon_t$ in state space form. What are $\boldsymbol{\Phi}$, $\mathbf{A}$, $\mathbf{Q}$, $R$?

**Q6.** Explain how the Kalman filter handles missing observations. Why does no imputation step exist?

**Q7.** What is the innovation form of the log-likelihood? Why is it called "exact" compared to conditional sum of squares?

**Q8.** In a structural model with level + seasonality, how does the Kalman filter simultaneously estimate both components? What is the role of the state vector?

---

### **Computation Questions**

**Q9.** For the local level model with $\sigma_w^2 = 0.25$, $\sigma_v^2 = 2.25$:
- What is the signal-to-noise ratio $Q/R$?
- Will the Kalman filter track observations closely or smooth heavily? Why?
- What happens to the Kalman gain as $\sigma_v^2 \to \infty$?

**Q10.** The steady-state Kalman gain for the local level model satisfies: $K = \frac{P}{P+R}$ where $P$ solves $P = \frac{(P+Q)R}{P+Q+R}$. For $Q = 1$, $R = 4$:
- Solve for the steady-state $P$ and $K$
- Interpret: what fraction of new information does the filter use?

**Q11.** You run a Kalman filter on 200 observations. At $t = 100$, a gap of 20 missing values begins. Sketch qualitatively what happens to $P_{t|t}$ during and after the gap.

---

### **Practical Challenge**

**Q12.** Modify the from-scratch Kalman filter code (Part 2) to implement the AR(1) + noise model with general $\phi$. Compare the filtered output for $\phi = 0.3$ vs $\phi = 0.95$.

**Q13.** Using `statsmodels.UnobservedComponents`, fit a local linear trend + seasonal model to monthly airline passenger data (or the simulated PM2.5 from Week 2). Extract the trend and seasonal components. Do the residuals pass Ljung–Box?
